# Pizza Hut Chatbot with SQLite Database Tools                                          

In [1]:
# Install Dependencies
!pip install langchain langchain-ollama gradio --quiet                                          

In [2]:
#Set up SQLite Database
import sqlite3
def get_connection():
    return sqlite3.connect("pizzahut.db")

con=get_connection()
cursor=con.cursor()

cursor.execute("DROP TABLE IF EXISTS menu")
cursor.execute("DROP TABLE IF EXISTS orders")


cursor.execute( """
CREATE TABLE  menu(
id INTEGER PRIMARY KEY,
name Text,
size Text,
price Real,
description Text)
"""
)

cursor.execute("""
CREATE TABLE  orders(
id INTEGER PRIMARY KEY AUTOINCREMENT,
item_name Text,
size Text,
quantity int,
price Real,
status Text,
created_at Text)
"""
)


cursor.execute("SELECT COUNT(*) FROM menu")
count = cursor.fetchone()[0]

if count == 0:
    menu_items = [
        ("Margherita","Small",8.0,"tomato sauce, mozzarella, basil"),
        ("Margherita","Large",14.0,"tomato sauce, mozzarella, basil"),
        ("Pepperoni","Small",10.0,"tomato sauce, mozzarella, pepperoni"),
        ("Pepperoni","Large",18.0,"tomato sauce, mozzarella, pepperoni"),
        ("Veggie","Medium",12.0,"bell peppers, olives, onions, mushrooms"),
        ("BBQ Chicken","Large",20.0,"BBQ sauce, chicken, red onions")
    ]
    
    cursor.executemany(
        "INSERT INTO menu (name,size,price,description) VALUES (?,?,?,?)",
        menu_items
    )

con.commit()
con.close()

print("Database initialized")


Database initialized


In [8]:
from langchain.tools import tool
from datetime import datetime

# Tool 1: Get Menu
@tool
def get_menu() -> str:
    """Fetches all available items from the Pizza Hut menu."""
    
    conn = get_connection()
    cursor = conn.cursor()
    
    cursor.execute("SELECT name,size,price,description FROM menu")
    rows = cursor.fetchall()
    
    conn.close()
    
    if not rows:
        return "Menu is empty."
    
    menu_text = "🍕 Pizza Hut Menu\n\n"
    
    for name,size,price,desc in rows:
        menu_text += f"{name} ({size}) - ${price}\n{desc}\n\n"
    
    return menu_text


# Tool 2: Place Order
@tool
def place_order(item_name:str, size:str, quantity:int) -> str:
    """Places an order for a menu item. Requires item name, size, and quantity."""
    
    conn = get_connection()
    cursor = conn.cursor()
    
    cursor.execute(
        "SELECT price FROM menu WHERE name=? AND size=?",
        (item_name,size)
    )
    
    row = cursor.fetchone()
    
    if not row:
        conn.close()
        return f"Sorry, {item_name} ({size}) is not available."
    
    price = row[0]
    total_price = price * quantity
    
    cursor.execute("""
        INSERT INTO orders (item_name,size,quantity,price,status,created_at)
        VALUES (?,?,?,?,?,?)
    """,(
        item_name,
        size,
        quantity,
        total_price,
        "Preparing",
        datetime.now().isoformat()
    ))
    
    conn.commit()
    conn.close()
    
    return f"✅ Order placed: {quantity} x {item_name} ({size})\nTotal: ${total_price}"


# Tool 3: Get Orders
@tool
def get_orders() -> str:
    """Retrieves all current orders placed."""
    
    conn = get_connection()
    cursor = conn.cursor()
    
    cursor.execute("SELECT item_name,size,quantity,price,status FROM orders")
    rows = cursor.fetchall()
    
    conn.close()
    
    if not rows:
        return "No orders yet."
    
    result = "📦 Current Orders\n\n"
    
    for item,size,qty,price,status in rows:
        result += f"{qty} x {item} ({size}) - ${price} [{status}]\n"
    
    return result

In [9]:
SYSTEM_PROMPT = """
You are a Pizza Hut ordering assistant.

Rules:
- Use get_menu to show available items
- Use place_order to place customer orders
- Use get_orders to show current orders
- Always confirm the order before placing it
- Always mention the price when showing menu items
"""

In [10]:
from langchain_ollama import ChatOllama
from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    AIMessage,
    ToolMessage
)

llm = ChatOllama(
    model="llama3.2",
    temperature=0.7
)

tools = [get_menu, place_order, get_orders]

llm_with_tools = llm.bind_tools(tools)

tool_map = {t.name: t for t in tools}

In [11]:
def run_agent(user_message, history):

    messages = [SystemMessage(content=SYSTEM_PROMPT)]
    
    # Convert history
    for msg in history:
        if msg["role"] == "user":
            messages.append(HumanMessage(content=msg["content"]))
        else:
            messages.append(AIMessage(content=msg["content"]))
    
    messages.append(HumanMessage(content=user_message))
    
    while True:
        
        ai_msg = llm_with_tools.invoke(messages)
        messages.append(ai_msg)
        
        if not ai_msg.tool_calls:
            return ai_msg.content
        
        for tool_call in ai_msg.tool_calls:
            
            tool_name = tool_call["name"]
            args = tool_call["args"]
            
            tool = tool_map[tool_name]
            
            result = tool.invoke(args)
            
            messages.append(
                ToolMessage(
                    content=str(result),
                    tool_call_id=tool_call["id"]
                )
            )

In [12]:
history = []

queries = [
    "What's on the menu?",
    "I'd like to order 2 large Pepperoni pizzas",
    "Show me my orders"
]

for q in queries:
    
    response = run_agent(q, history)
    
    history.append({"role":"user","content":q})
    history.append({"role":"assistant","content":response})
    
    print("User:", q)
    print("Bot:", response)
    print()

User: What's on the menu?
Bot: Here is the menu for Pizza Hut:

- Margherita (Small): $8.0
- Margherita (Large): $14.0
- Pepperoni (Small): $10.0
- Pepperoni (Large): $18.0
- Veggie (Medium): $12.0
- BBQ Chicken (Large): $20.0

Would you like to place an order?

User: I'd like to order 2 large Pepperoni pizzas
Bot: Here are the available options for sides and desserts:

- Breadsticks (Medium): $6.0
- Breadsticks (Large): $8.0
- Side Salad (Medium): $4.0
- Side Salad (Large): $6.0
- Garlic Knots (Medium): $5.0
- Garlic Knots (Large): $7.0
- Chocolate Fudge Brownie: $5.0
- Fruit Cup: $4.0

Which item would you like to add to your order?

User: Show me my orders
Bot: Here's a summary of your order:

You have placed an order for:

2 x Large Pepperoni Pizzas

The total cost of your order is: $36.0

Please confirm if this is correct before we proceed with preparation and payment.



In [13]:
import gradio as gr

history = []

def chat(message, chat_history):

    response = run_agent(message, history)
    
    history.append({"role":"user","content":message})
    history.append({"role":"assistant","content":response})

    chat_history.append({"role":"user","content":message})
    chat_history.append({"role":"assistant","content":response})
    
    
    return "", chat_history


with gr.Blocks(title="Pizza Hut Order Bot") as demo:
    
    gr.Markdown("# 🍕 Pizza Hut Order Bot")
    
    chatbot = gr.Chatbot(height=450)
    
    msg = gr.Textbox(
        placeholder="Ask for the menu or order pizza..."
    )
    
    clear = gr.Button("Clear Chat")
    
    msg.submit(chat, [msg, chatbot], [msg, chatbot])
    
    clear.click(lambda: None, None, chatbot, queue=False)


demo.launch(theme=gr.themes.Soft())

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
